In [ ]:
### Objective
#The objective of this project is to analyze the relationship between market sentiment (Fear vs Greed) and trader behavior using historical trading data. 
#The goal is to identify patterns in performance, trading activity, and risk-taking, and derive actionable strategies based on market conditions.

import os
os.listdir()

In [29]:
import pandas as pd
sentiment = pd.read_csv("fear_greed_index.csv")
trader = pd.read_csv("historical_data.csv")
sentiment.head()
trader.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [41]:
# shape
print("Sentiment shape:", sentiment.shape)
print("Trader shape:", trader.shape)

# columns
print("\nSentiment columns:", sentiment.columns)
print("\nTrader columns:", trader.columns)

# missing values
print("\nMissing values (Sentiment):\n", sentiment.isnull().sum())
print("\nMissing values (Trader):\n", trader.isnull().sum())

# duplicates
print("\nDuplicates (Sentiment):", sentiment.duplicated().sum())
print("Duplicates (Trader):", trader.duplicated().sum())

Sentiment shape: (2644, 5)
Trader shape: (211224, 17)

Sentiment columns: Index(['timestamp', 'value', 'classification', 'date', 'Date'], dtype='object')

Trader columns: Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp', 'Date'],
      dtype='object')

Missing values (Sentiment):
 timestamp         0
value             0
classification    0
date              0
Date              0
dtype: int64

Missing values (Trader):
 Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
Date                0
dtype: int64

Du

In [31]:
# Data cleaning
trader['Timestamp IST'] = pd.to_datetime(trader['Timestamp IST'], dayfirst=True)
# extract only date
trader['Date'] = trader['Timestamp IST'].dt.date
# convert sentiment date
sentiment['Date'] = pd.to_datetime(sentiment['date']).dt.date

In [42]:
merged = pd.merge(trader, sentiment, on='Date', how='inner')
merged.head()
print("Merged shape:", merged.shape)
print(merged.columns)

Merged shape: (211218, 21)
Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp', 'Date', 'timestamp', 'value', 'classification', 'date'],
      dtype='object')


In [34]:
daily_pnl = merged.groupby('Date')['Closed PnL'].sum().reset_index()
trades_per_day = merged.groupby('Date').size().reset_index(name='trade_count')
avg_size = merged.groupby('Date')['Size USD'].mean().reset_index()

In [35]:
# create profit flag
merged['is_profit'] = merged['Closed PnL'] > 0

# calculate win rate
win_rate = merged.groupby('Date')['is_profit'].mean().reset_index()

In [36]:
long_short = merged.groupby(['Date', 'Side']).size().unstack(fill_value=0)
long_short.head()

Side,BUY,SELL
Date,,
2023-05-01,3,0
2023-12-05,7,2
2023-12-14,5,6
2023-12-15,2,0
2023-12-16,3,0


In [37]:
win_sentiment = pd.merge(win_rate, sentiment[['Date','classification']], on='Date')
win_summary = win_sentiment.groupby('classification')['is_profit'].mean().reset_index()
win_summary.rename(columns={'is_profit': 'win_rate'}, inplace=True)
win_summary = win_summary.sort_values(by='win_rate', ascending=False)
win_summary

,classification,win_rate
1,Extreme Greed,0.467424
3,Greed,0.335986
4,Neutral,0.331886
2,Fear,0.329112
0,Extreme Fear,0.327341


In [38]:
trades_sentiment = pd.merge(trades_per_day, sentiment[['Date','classification']], on='Date')
trade_summary = trades_sentiment.groupby('classification')['trade_count'].mean().reset_index()
trade_summary = trade_summary.sort_values(by='trade_count', ascending=False)
trade_summary

,classification,trade_count
0,Extreme Fear,1528.571429
2,Fear,679.527473
4,Neutral,562.477612
1,Extreme Greed,350.807018
3,Greed,260.637306


In [39]:
pnl_sentiment = pd.merge(daily_pnl, sentiment[['Date','classification']], on='Date')
pnl_summary = pnl_sentiment.groupby('classification')['Closed PnL'].mean().reset_index()
pnl_summary = pnl_summary.sort_values(by='Closed PnL', ascending=False)
pnl_summary

,classification,Closed PnL
0,Extreme Fear,52793.589178
2,Fear,36891.818040
1,Extreme Greed,23817.292199
4,Neutral,19297.323516
3,Greed,11140.566181


In [40]:
print("Win Rate by Sentiment:\n", win_summary)
print("\nTrade Activity:\n", trade_summary)
print("\nAverage PnL:\n", pnl_summary)

Win Rate by Sentiment:
   classification  win_rate
1  Extreme Greed  0.467424
3          Greed  0.335986
4        Neutral  0.331886
2           Fear  0.329112
0   Extreme Fear  0.327341

Trade Activity:
   classification  trade_count
0   Extreme Fear  1528.571429
2           Fear   679.527473
4        Neutral   562.477612
1  Extreme Greed   350.807018
3          Greed   260.637306

Average PnL:
   classification    Closed PnL
0   Extreme Fear  52793.589178
2           Fear  36891.818040
1  Extreme Greed  23817.292199
4        Neutral  19297.323516
3          Greed  11140.566181


In [44]:
# merge segmentation back to main data
merged_segment = pd.merge(merged, trades_per_account[['Account','type']], on='Account')

# compare PnL
merged_segment.groupby('type')['Closed PnL'].mean()

# compare trade size
merged_segment.groupby('type')['Size USD'].mean()

type
Frequent      5800.826314
Infrequent    4393.623035
Name: Size USD, dtype: float64

In [ ]:
#SUMMARY

# Insight 1: Win Rate vs Sentiment
# Traders achieve the highest win rate during Extreme Greed (~46.7%),
# indicating that strong bullish sentiment leads to more profitable trades.
# This suggests that clear upward trends improve trading success and consistency.

# Insight 2: Trading Activity vs Sentiment
# Trading activity is highest during Extreme Fear, suggesting reactive or
# panic-driven behavior during market downturns. Traders tend to execute more
# trades in volatile conditions due to uncertainty and rapid price movements.

# Insight 3: Profitability vs Sentiment
# Average PnL is highest during Extreme Fear, indicating that high volatility
# creates opportunities for larger profits despite lower win rates. This shows
# that while success rate is lower, profit potential per trade is higher.

# Insight 4: Trader Segmentation (Frequent vs Infrequent)
# Frequent traders execute trades with a higher average trade size (~5800 USD)
# compared to infrequent traders (~4393 USD). This indicates that frequent traders
# take larger positions and are more actively involved in the market.

# Frequent traders are more aggressive and risk-taking, while infrequent traders
# are more conservative. Trading frequency is strongly linked to risk exposure
# and capital allocation.

In [ ]:
## Strategy Recommendations
#1. During Extreme Fear, traders can take advantage of volatility but must apply strong risk management due to lower win rates.
#2. During Extreme Greed, traders can focus on consistent profits by following market trends, while avoiding overconfidence and excessive risk-taking.

In [ ]:
# Conclusion
# Market sentiment plays a critical role in shaping trader behavior and performance.
# While Extreme Greed conditions provide higher win rates and more predictable trends,
# Extreme Fear conditions offer higher profit potential due to increased volatility.
# Therefore, traders should adopt a dynamic strategy—being opportunity-driven in
# volatile markets and consistency-focused in trending markets—while always maintaining
# disciplined risk management.